In [1]:
import gc, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from itertools import combinations
import torch
from tqdm.auto import tqdm
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Apr 1, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA_V2
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir
from src.fully_connected_colab import (
    compute_batch_size,
    detect_device,
    prepare_gpu_data,
    save_colab_run,
    train_one_model,
)

Mounted at /content/drive


## Load data

In [2]:
DATA_SET = 'rand_D'

df_train = pd.read_parquet(CLEAN_DATA_V2 / (DATA_SET + '_train_v2.parquet'))
df_val   = pd.read_parquet(CLEAN_DATA_V2 / (DATA_SET + '_val_v2.parquet'))
df_test  = pd.read_parquet(CLEAN_DATA_V2 / (DATA_SET + '_test_v2.parquet'))


OUTPUT_DIR = make_run_dir()

## GPU auto-detect

In [3]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)

L4  |  VRAM: 24 GB  |  MAX_BATCH=32,768  |  dtype=torch.bfloat16


## Hyperparameters

In [4]:
SEED = 42
MAX_EPOCHS = 100
PATIENCE = 30
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
NEURONS = 80
HIDDEN_LAYERS = 3

BATCH_SIZE = compute_batch_size(len(df_train), CFG['MAX_BATCH'])
BASE_LR = 1e-3
BASE_BATCH = 4096
INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH) ** 0.5

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

steps_per_epoch = len(df_train) // BATCH_SIZE
print(f'MAX_BATCH={CFG["MAX_BATCH"]:,}  adaptive BATCH_SIZE={BATCH_SIZE:,}  '
      f'INIT_LR={INIT_LR:.6f}  n_train={len(df_train):,}  '
      f'steps/epoch~{steps_per_epoch}')
print(f'MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}  '
      f'WARMUP={WARMUP_EPOCHS} epochs')

MAX_BATCH=32,768  adaptive BATCH_SIZE=16,384  INIT_LR=0.002000  n_train=843,153  steps/epoch~51
MAX_EPOCHS=100  PATIENCE=30  WARMUP=5 epochs


## Feature definitions

In [5]:
FEATURES_3 = ['delta', 'T', 'spy_ret']
EXTRA_FEATURES = ['iv_lag', 'vix_lag', 'vix_mom_lag', 'vix_mom', 'gamma', 'theta', 'vega', 'rho']
ALL_FEATURES = FEATURES_3 + EXTRA_FEATURES
TARGET = 'd_iv'

# Generate ALL 2^8 = 256 combinations of EXTRA_FEATURES (0 to 8 extras)
feature_combos = []
feature_combos.append(('3F', FEATURES_3.copy()))  # Base case

for r in range(1, len(EXTRA_FEATURES) + 1):
    for combo in combinations(EXTRA_FEATURES, r):
        name = '3F+' + '+'.join(combo)
        feats = FEATURES_3 + list(combo)
        feature_combos.append((name, feats))

print(f'Total feature combinations: {len(feature_combos)}')
print(f'  Base (3F):  1')
for r in range(1, len(EXTRA_FEATURES) + 1):
    n = len(list(combinations(EXTRA_FEATURES, r)))
    print(f'  3F + {r}:     {n}')

Total feature combinations: 256
  Base (3F):  1
  3F + 1:     8
  3F + 2:     28
  3F + 3:     56
  3F + 4:     70
  3F + 5:     56
  3F + 6:     28
  3F + 7:     8
  3F + 8:     1


## Pre-allocate on GPU

In [6]:
gpu_data = prepare_gpu_data(df_train, df_val, df_test, ALL_FEATURES, TARGET, DEVICE)
COL_IDX = gpu_data['col_idx']

Data on GPU  |  VRAM used: 0.28 GB / 24 GB  |  Free: 23.4 GB
Train: 843,153  Val: 240,901  Test: 120,451  Features: 11


## Analytic benchmark

In [7]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

Analytic Benchmark
SSE = 8.3745  RMSE = 0.008338
Coefficients: a = -0.165665, b = -0.000516, c = 0.032673


## Train all feature combinations

In [8]:
train_kw = dict(
    Xtr=gpu_data['Xtr'],
    Xva=gpu_data['Xva'],
    Xte=gpu_data['Xte'],
    ytr=gpu_data['ytr'],
    yva=gpu_data['yva'],
    y_test=gpu_data['y_test'],
    hw_sse=hw['sse'],
    all_feature_names=ALL_FEATURES,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    use_amp=USE_AMP,
    nan_mask_tr=gpu_data['nan_mask_tr'],
    nan_mask_va=gpu_data['nan_mask_va'],
    nan_mask_ytr=gpu_data['nan_mask_ytr'],
    nan_mask_yva=gpu_data['nan_mask_yva'],
    seed=SEED,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    lr_patience=LR_PATIENCE,
    lr_factor=LR_FACTOR,
    init_lr=INIT_LR,
    warmup_epochs=WARMUP_EPOCHS,
    neurons=NEURONS,
    hidden_layers=HIDDEN_LAYERS,
)

sep = "=" * 70
print('\n' + sep)
print(f'  TRAINING {len(feature_combos)} MODELS')
print(f'  GPU: {CFG["GPU"]}  |  Batch: {BATCH_SIZE:,}  |  AMP: {USE_AMP}  |  Max epochs: {MAX_EPOCHS}')
print(sep + '\n')

all_results = {}
t_global = time.time()

pbar = tqdm(feature_combos, desc='Feature sweep', unit='model')
for name, feats in pbar:
    cols = [COL_IDX[f] for f in feats]
    result = train_one_model(name, cols, **train_kw)
    all_results[name] = result
    pbar.set_postfix_str(
        f'{name}  SSE={result["sse"]:.4f}  Gain={result["gain_vs_hw"]:+.1f}%'
    )

elapsed_s = time.time() - t_global
print(f'\nDone: {elapsed_s / 60:.1f} min for {len(all_results)} models '
      f'(avg {elapsed_s / max(len(all_results), 1):.1f}s/model)')


  TRAINING 256 MODELS
  GPU: L4  |  Batch: 16,384  |  AMP: True  |  Max epochs: 100



Feature sweep:   0%|          | 0/256 [00:00<?, ?model/s]


Done: 52.3 min for 256 models (avg 12.2s/model)


## Results summary

In [9]:
summary, df_results = save_colab_run(
    OUTPUT_DIR,
    y_test=gpu_data['y_test'],
    hw=hw,
    models=all_results,
)
summary

,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,8.374537,0.000070,0.008338,0.003523,0.000748,0.001885,0.125011,None,None,None
1,3F,10.975561,0.000091,0.009546,0.005470,-0.000109,0.003735,-0.146749,13.1s,-31.06%,None
2,3F+iv_lag,16.846130,0.000140,0.011826,0.008326,-0.000327,0.006169,-0.760118,12.5s,-101.16%,-53.49%
3,3F+vix_lag,20.193176,0.000168,0.012948,0.008821,-0.000484,0.006409,-1.109824,12.3s,-141.13%,-19.87%
4,3F+vix_mom_lag,20.058388,0.000167,0.012905,0.008799,-0.000897,0.006396,-1.095741,12.6s,-139.52%,0.67%
...,...,...,...,...,...,...,...,...,...,...,...
253,3F+iv_lag+vix_lag+vix_mom+gamma+theta+vega+rho,8.688316,0.000072,0.008493,0.006176,0.001744,0.004703,0.092227,12.5s,-3.75%,-31.83%
254,3F+iv_lag+vix_mom_lag+vix_mom+gamma+theta+vega...,8.170237,0.000068,0.008236,0.005900,-0.000794,0.004430,0.146357,12.2s,2.44%,5.96%
255,3F+vix_lag+vix_mom_lag+vix_mom+gamma+theta+veg...,9.401947,0.000078,0.008835,0.005696,-0.000421,0.004146,0.017665,11.9s,-12.27%,-15.08%
256,3F+iv_lag+vix_lag+vix_mom_lag+vix_mom+gamma+th...,7.343217,0.000061,0.007808,0.005491,-0.000233,0.003997,0.232766,11.9s,12.31%,21.90%


## Top 10 by Gain vs Hull-White

In [10]:
print('Top 10 feature combos by Gain vs Hull-White:\n')
top10 = df_results.head(10)[['combo_name', 'n_features', 'SSE', 'RMSE', 'Gain_vs_HW_%', 'training_time_s']].copy()
top10['Gain_vs_HW_%'] = top10['Gain_vs_HW_%'].round(2)
top10['RMSE'] = top10['RMSE'].round(6)
top10['SSE'] = top10['SSE'].round(4)
top10['training_time_s'] = top10['training_time_s'].round(1)
top10

Top 10 feature combos by Gain vs Hull-White:



,combo_name,n_features,SSE,RMSE,Gain_vs_HW_%,training_time_s
0,3F+iv_lag+vix_lag+vix_mom_lag+vix_mom+gamma+th...,10,4.8949,0.006375,41.55,12.5
1,3F+iv_lag+vix_lag+vix_mom_lag+vix_mom+gamma+th...,10,4.9578,0.006416,40.80,12.2
2,3F+iv_lag+vix_lag+vix_mom_lag+vix_mom+gamma+ve...,10,5.7586,0.006914,31.24,11.9
3,3F+iv_lag+vix_lag+vix_mom_lag+vix_mom+theta+ve...,10,5.8053,0.006942,30.68,12.0
4,3F+iv_lag+vix_lag+vix_mom_lag+gamma+theta,8,6.5012,0.007347,22.37,12.4
5,3F+iv_lag+vix_lag+vix_mom_lag+gamma+theta+vega...,10,6.5907,0.007397,21.30,12.0
6,3F+iv_lag+vix_lag+gamma+theta+vega,8,7.1647,0.007712,14.45,12.1
7,3F+iv_lag+vix_mom+gamma+theta+vega,8,7.2766,0.007772,13.11,12.3
8,3F+iv_lag+vix_mom_lag+vix_mom+gamma+theta,8,7.2837,0.007776,13.03,12.1
9,3F+iv_lag+vix_lag+vix_mom_lag+vix_mom+gamma+th...,11,7.3432,0.007808,12.31,11.9


## Best per group (3F through 3F+8)

In [11]:
group_labels = [('3F (base)', 3)] + [(f'+{k} ({3+k}F)', 3+k) for k in range(1, len(EXTRA_FEATURES) + 1)]
for nf_label, n in group_labels:
    sub = df_results[df_results['n_features'] == n]
    if len(sub) == 0:
        continue
    best = sub.iloc[0]
    print(f"{nf_label}: {best['combo_name']}")
    print(f"    SSE={best['SSE']:.4f}  RMSE={best['RMSE']:.6f}  Gain={best['Gain_vs_HW_%']:.2f}%\n")

3F (base): 3F
    SSE=10.9756  RMSE=0.009546  Gain=-31.06%

+1 (4F): 3F+gamma
    SSE=14.9572  RMSE=0.011143  Gain=-78.60%

+2 (5F): 3F+iv_lag+vix_lag
    SSE=9.4697  RMSE=0.008867  Gain=-13.08%

+3 (6F): 3F+iv_lag+gamma+rho
    SSE=8.4431  RMSE=0.008372  Gain=-0.82%

+4 (7F): 3F+iv_lag+vix_mom+gamma+theta
    SSE=7.3715  RMSE=0.007823  Gain=11.98%

+5 (8F): 3F+iv_lag+vix_lag+vix_mom_lag+gamma+theta
    SSE=6.5012  RMSE=0.007347  Gain=22.37%

+6 (9F): 3F+iv_lag+vix_mom_lag+vix_mom+gamma+theta+vega
    SSE=8.5978  RMSE=0.008449  Gain=-2.67%

+7 (10F): 3F+iv_lag+vix_lag+vix_mom_lag+vix_mom+gamma+theta+vega
    SSE=4.8949  RMSE=0.006375  Gain=41.55%

+8 (11F): 3F+iv_lag+vix_lag+vix_mom_lag+vix_mom+gamma+theta+vega+rho
    SSE=7.3432  RMSE=0.007808  Gain=12.31%



## Summary statistics

In [12]:
total_time_s = df_results['training_time_s'].sum()
print(f'Total training time: {total_time_s / 60:.1f} min ({total_time_s / 3600:.2f} hr)')
print(f'Models trained: {len(df_results)}')
print(f'Best overall: {df_results.iloc[0]["combo_name"]} (Gain={df_results.iloc[0]["Gain_vs_HW_%"]:.2f}%)')

Total training time: 52.1 min (0.87 hr)
Models trained: 256
Best overall: 3F+iv_lag+vix_lag+vix_mom_lag+vix_mom+gamma+theta+vega (Gain=41.55%)


## Cleanup

In [13]:
del all_results, gpu_data
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Total training time: {total_time_s / 60:.1f} min for {len(df_results)} models')

if IN_COLAB:
    runtime.unassign()

Final VRAM: 0.39 GB / 24 GB
Total training time: 52.1 min for 256 models
